# RetailMind AI — Day 12: a classical sentiment baseline

Is a product review positive or negative? We clean Amazon reviews, turn them into numbers with TF-IDF, and train a logistic regression. This is the cheap baseline that Day 13's BERT has to beat.

Training takes about a minute, so this notebook **reads the saved results** from `python -m src.train_day12` and tries the saved model live on a few made-up reviews.

In [1]:
# Work from the project root so the `src` package can be imported
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: day3-4


## 1. Cleaning a review

In [2]:
from src.review_data import clean_text

messy = "Great sound!<br /><br />Bought   it at https://example.com/deal   and love it."
print(repr(messy))
print(repr(clean_text(messy)))

'Great sound!<br /><br />Bought   it at https://example.com/deal   and love it.'
'Great sound! Bought it at and love it.'


## 2. The results (from `python -m src.train_day12`)

In [3]:
import json
day12 = json.load(open("models/day12_results.json"))
print(day12["data"])
print("training reviews:", day12["train_reviews"], "| test reviews:", day12["test_reviews"], "| features:", day12["features"])
s = day12["test_scores"]
print(f"test accuracy {s['accuracy']} | precision {s['precision']} | recall {s['recall']} | F1 {s['f1']}")
print("accuracy on its own training reviews:", day12["train_accuracy"])
print("confusion matrix:", s["confusion_matrix"])

Amazon reviews (fancyzhx/amazon_polarity test file), balanced sample
training reviews: 4000 | test reviews: 1000 | features: 37263
test accuracy 0.87 | precision 0.8865 | recall 0.8629 | F1 0.8745
accuracy on its own training reviews: 0.9962
confusion matrix: {'negative_right': 417, 'negative_called_positive': 58, 'positive_called_negative': 72, 'positive_right': 453}


## 3. Which words matter? (the model's biggest weights)

In [4]:
print("pushes toward negative:", ", ".join(day12["strongest_words"]["negative"]))
print("pushes toward positive:", ", ".join(day12["strongest_words"]["positive"]))

pushes toward negative: not, poor, money, boring, bad, disappointed, worst, waste, nothing, disappointing
pushes toward positive: great, love, best, perfect, wonderful, excellent, well, awesome, the best, easy


## 4. Try the saved model on new reviews

In [5]:
import joblib
from src.config import SENTIMENT_TFIDF_FILE
from src.review_data import SENTIMENTS

model = joblib.load(SENTIMENT_TFIDF_FILE)
reviews = ["Absolutely love it, works perfectly and arrived early.",
           "Total waste of money. Stopped working after two days.",
           "It is not bad at all, quite good actually.",
           "Not good. Not worth the price."]
for review, probability in zip(reviews, model.predict_proba(reviews)[:, 1]):
    print(f"{SENTIMENTS[int(probability > 0.5)]:<9} (P positive {probability:.2f})  {review}")

positive  (P positive 0.98)  Absolutely love it, works perfectly and arrived early.
negative  (P positive 0.04)  Total waste of money. Stopped working after two days.
negative  (P positive 0.19)  It is not bad at all, quite good actually.
negative  (P positive 0.17)  Not good. Not worth the price.


## 5. Conclusion

- **87.0% accuracy on 1,000 unseen reviews**, with a margin of about plus or minus 1 point.
- **It memorises its training reviews** (99.6% on them), so it is overfitting; the regularisation strength was not tuned.
- It counts words, so negation ("not bad") and sarcasm are weak spots.
- This is the number Day 13 must beat to justify a much bigger model.